# LLM Web Search — Provider-Native Grounding

Let the model fetch live web results on its own — no scraping, no Serp API. Each provider exposes a **server-side search tool** that Anthropic/OpenAI run internally before generating the answer.

> **Providers**: OpenAI and Anthropic only (Gemini and Ollama excluded by design).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math

pd.set_option('display.max_columns',100)
pd.set_option('display.max_rows',100)

plt.style.use('dark_background')

import warnings
warnings.filterwarnings('ignore')

# Shared course helpers (msds565_helpers.py lives in the repo root).
# Notebooks sit two folders below the root, so '../..' points back to it.
import sys
sys.path.append('../..')
import msds565_helpers as helpers

import os

SEARCH_PROMPT = 'What are the latest developments in large language model research from the past week?'

---
## OpenAI

Uses the **Responses API** (`client.responses.create`) with the `web_search_preview` built-in tool. The model searches the web, then synthesizes an answer — all in one call.

In [ ]:
from openai import OpenAI

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
OPENAI_MODEL   = 'gpt-5'

openai_client = OpenAI(api_key=OPENAI_API_KEY)
openai_resp = openai_client.responses.create(
    model=OPENAI_MODEL,
    tools=[{'type': 'web_search_preview'}],
    input=SEARCH_PROMPT
)

for item in openai_resp.output:
    if item.type == 'message':
        for block in item.content:
            if block.type == 'output_text':
                print(block.text)

---
## Anthropic

Declare `web_search_20260209` as a tool; Anthropic's infrastructure executes the search and returns grounded results. The model cites sources in its reply.

In [ ]:
from anthropic import Anthropic

ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')
ANTHROPIC_MODEL   = 'claude-sonnet-4-5-20250929'

anthropic_client = Anthropic(api_key=ANTHROPIC_API_KEY)
anthropic_resp = anthropic_client.messages.create(
    model=ANTHROPIC_MODEL,
    max_tokens=1024,
    tools=[{'type': 'web_search_20260209', 'name': 'web_search'}],
    messages=[{'role': 'user', 'content': SEARCH_PROMPT}]
)

for block in anthropic_resp.content:
    if block.type == 'text':
        print(block.text)

## Review

**Takeaways**

- **Provider-native search runs server-side, before generation.** You do not fetch anything or manage an index; the model retrieves and then answers in one call.
- **That is the tradeoff against RAG.** Native search is far less work and gives current, general information; RAG searches *your* documents and lets you see exactly what was retrieved. They solve different problems.
- **Citations are the reason to prefer a grounded answer** over the model's training knowledge, because they are what make a claim checkable.
- **Not every provider offers it**, and the ones that do gate it behind different tool types and versions - which is also why the pinned versions in this notebook will age.